import the libraries important for data cleaning and preparation

In [1]:
import pandas as pd
import camelot
import warnings

some tweaks to supress irrelevant warnings and optimize some settings

In [5]:
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 5000)
pd.set_option('display.max_columns',  None)

Convert the pdf into csv

In [7]:
tables = camelot.read_pdf('../data/auction5.pdf', pages='all', flavor='lattice')
combined = pd.concat([table.df for table in tables], ignore_index=True)
combined.to_csv('../data/clean_auction5.csv')

import the csv file for manipulation and cleaning give the dataset the correct columns name

In [13]:
df = pd.read_csv('../data/clean_auction5.csv')
df.columns = ['one', 'no', 'rank', 'winners', 'price_per_sqm', 'down_payment_pct', 'code', 'district', 'sqm',  'comment']
print(f"The shape of the first dataset is {df.shape}")

The shape of the first dataset is (687, 10)


In [20]:
df.head(700)

,price_per_sqm,down_payment_pct,subcity,district,sqm
0,"26,531.22",100%,Lemi Kura,04,231
1,"31,775.00",70%,Lemi Kura,NaN,NaN
2,"21,157.00",100%,Lemi Kura,NaN,NaN
3,"32,621.00",72%,Lemi Kura,04,249
4,"18,885.00",100%,Lemi Kura,NaN,NaN
5,"15,000.00",50%,Lemi Kura,NaN,NaN
6,"45,475.50",100%,Lemi Kura,04,110
7,"39,228.00",100%,Lemi Kura,NaN,NaN
8,"39,051.00",100%,Lemi Kura,NaN,NaN
9,"31,150.00",100%,Lemi Kura,04,130


drop irrelevant columns

In [16]:
df.drop(columns=['one', 'no', 'rank', 'winners', 'code', 'comment'], inplace=True)

drop irrelevant and empty rows, then rearrange the index

In [17]:
df = df.dropna(subset=['price_per_sqm'])
df = df[df['price_per_sqm'].astype(str).str.contains(r'\d', regex=True, na=False)]
df = df.reset_index(drop=True)

rename the subcity names into the appropriate kinda names

In [19]:
df.loc[0:284, 'subcity'] = 'Lemi Kura'
df.loc[285:398, 'subcity'] = 'Akaki Kality'
df.loc[399:470, 'subcity'] = 'Nefas - Silk Lafto'
df.loc[471:512, 'subcity'] = 'Yeka'
df.loc[513:533, 'subcity'] = 'Gulele'
df.loc[534:557, 'subcity'] = 'Addis Ketema'
df.loc[558:, 'subcity'] = 'Kolfe Keranyo'

col_to_move = df.pop(df.columns[4])
df.insert(2, 'subcity', col_to_move)

inspect the data types and change them to the appropriate ones

In [21]:
df['price_per_sqm'] = df['price_per_sqm'].astype(str).str.replace(',', '').str.strip().astype('float64')
df['down_payment_pct'] = df['down_payment_pct'].astype(str).str.replace('%', '').str.strip().astype('float64')
df['district'] = df['district'].ffill(limit=2).astype('float64')
df['sqm'] = df['sqm'].ffill(limit=2).astype('float64')
display(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 648 entries, 0 to 647
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   price_per_sqm     648 non-null    float64
 1   down_payment_pct  648 non-null    float64
 2   subcity           648 non-null    str    
 3   district          640 non-null    float64
 4   sqm               640 non-null    float64
dtypes: float64(4), str(1)
memory usage: 25.4 KB


None

In [22]:
df.to_csv('../data/clean_auction5.csv', index=False)